In [ ]:
BEAM_WIDTH = 2**24
START_PUZZLE_ID = 0
PUZZLE_COUNT = 1
DEPTH_LIMIT = 60
GITHUB_REPO_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
GITHUB_BRANCH = "main"
ENABLE_DEBUG = True
ENABLE_DEPTH_LOGS = True
ENABLE_DEBUG_LOGS = False
DEBUG_STREAM_TIMING = False
DEBUG_INFERENCE_TRACE = False
DEBUG_PATH_TRACE = False
DEBUG_FINAL_VALIDATE = False
DEPTH_LOG_EVERY = 1
PUZZLE_LOG_EVERY = 1
HISTORY_MODE = "ram"
HISTORY_SLOT_COUNT = 3
HISTORY_WORKERS = 1
STOP_ON_FAILURE = False

RUNTIME_CONFIG_MODE = "manual"
SHARD_BUFFER_COUNT = 2
STREAM3_RING_SLOTS = 1
SHARD_COUNT = 4
STREAM4_BATCH_CANDIDATES = 196608
STREAM4_TRIGGER_CANDIDATES = 786432
SHARD_CAPACITY_CANDIDATES = BEAM_WIDTH // SHARD_COUNT
STREAM4_ACTIVE_SORT_SLOTS = 2
GLOBAL_SPILL_CAPACITY = 0


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

WORK_DIR = Path('/kaggle/working')
TMP_DIR = Path('/tmp')
REPO_DIR = TMP_DIR / 'beam_solver'
CUTLASS_DIR = TMP_DIR / 'cutlass'
BUILD_DIR = TMP_DIR / 'beam_build'

def run_checked(cmd, cwd=None, env=None):
    print('+ ' + ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

for transient_dir in (REPO_DIR, BUILD_DIR):
    if transient_dir.exists():
        shutil.rmtree(transient_dir)
run_checked(['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPO_URL, REPO_DIR])

if not (CUTLASS_DIR / 'include').exists():
    if CUTLASS_DIR.exists():
        shutil.rmtree(CUTLASS_DIR)
    run_checked(['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/cutlass.git', CUTLASS_DIR])

depth_logs = 'ON' if ENABLE_DEPTH_LOGS else 'OFF'
debug_logs = 'ON' if ENABLE_DEBUG_LOGS else 'OFF'
debug_master = 'ON' if ENABLE_DEBUG else 'OFF'
debug_stream_timing = 'ON' if DEBUG_STREAM_TIMING else 'OFF'
debug_inference_trace = 'ON' if DEBUG_INFERENCE_TRACE else 'OFF'
debug_path_trace = 'ON' if DEBUG_PATH_TRACE else 'OFF'
debug_final_validate = 'ON' if DEBUG_FINAL_VALIDATE else 'OFF'
run_checked([
    'cmake', '-S', REPO_DIR, '-B', BUILD_DIR, '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    f'-DCUTLASS_DIR={CUTLASS_DIR}',
    f'-DBEAM_ENABLE_DEBUG={debug_master}',
    f'-DBEAM_ENABLE_DEPTH_LOGS={depth_logs}',
    f'-DBEAM_ENABLE_DEBUG_LOGS={debug_logs}',
    f'-DBEAM_DEBUG_STREAM_TIMING={debug_stream_timing}',
    f'-DBEAM_DEBUG_INFERENCE_TRACE={debug_inference_trace}',
    f'-DBEAM_DEBUG_PATH_TRACE={debug_path_trace}',
    f'-DBEAM_DEBUG_FINAL_VALIDATE={debug_final_validate}',
])
run_checked(['cmake', '--build', BUILD_DIR, '--target', 'production_runner', '-j', '2'])


In [ ]:
import re
import time
import pandas as pd

KNOWN_SOLUTION_PATHS = {}


LOG_DIR = WORK_DIR / 'run_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = WORK_DIR / 'beam_run_results.csv'
SUBMISSION_CSV = WORK_DIR / 'submission.csv'
SOLVED_RE = re.compile(r'puzzle_solved=(\d+) puzzle_id=(\d+) seconds=([0-9.eE+-]+) solution_length=(-?\d+) solution=(.*)$')

def run_puzzle(puzzle_id: int, puzzle_index: int):
    env = os.environ.copy()
    env['BEAM_HISTORY_MODE'] = HISTORY_MODE
    env['BEAM_HISTORY_SLOT_COUNT'] = str(HISTORY_SLOT_COUNT)
    env['BEAM_HISTORY_WORKERS'] = str(HISTORY_WORKERS)
    env['BEAM_DEPTH_LOG_EVERY'] = str(DEPTH_LOG_EVERY)
    env['BEAM_WEIGHT_DIR'] = str(REPO_DIR / 'stream1_weights')
    env['BEAM_RUNTIME_CONFIG_MODE'] = RUNTIME_CONFIG_MODE
    env['BEAM_SHARD_BUFFER_COUNT'] = str(SHARD_BUFFER_COUNT)
    env['BEAM_STREAM3_RING_SLOTS'] = str(STREAM3_RING_SLOTS)
    env['BEAM_SHARD_COUNT'] = str(SHARD_COUNT)
    env['BEAM_STREAM4_BATCH_CANDIDATES'] = str(STREAM4_BATCH_CANDIDATES)
    env['BEAM_STREAM4_TRIGGER_CANDIDATES'] = str(STREAM4_TRIGGER_CANDIDATES)
    env['BEAM_SHARD_CAPACITY_CANDIDATES'] = str(SHARD_CAPACITY_CANDIDATES)
    env['BEAM_STREAM4_ACTIVE_SORT_SLOTS'] = str(STREAM4_ACTIVE_SORT_SLOTS)
    env['BEAM_GLOBAL_SPILL_CAPACITY'] = str(GLOBAL_SPILL_CAPACITY)
    cmd = [str(BUILD_DIR / 'production_runner'), str(puzzle_id), str(DEPTH_LIMIT), str(BEAM_WIDTH)]
    log_path = LOG_DIR / f'puzzle_{puzzle_id}.log'
    parsed = None
    started = time.perf_counter()
    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            log.write(line)
            log.flush()
            line = line.rstrip('\n')
            match = SOLVED_RE.search(line)
            if match:
                parsed = match
                print(line)
            elif ENABLE_DEPTH_LOGS or line.startswith('track_solution_'):
                print(line)
        code = proc.wait()
    elapsed = time.perf_counter() - started
    if code != 0:
        result = {'puzzle_id': puzzle_id, 'solved': 0, 'seconds': elapsed, 'length': None, 'solution': '', 'return_code': code, 'log_path': str(log_path)}
        if STOP_ON_FAILURE:
            raise RuntimeError(f'production_runner failed: puzzle_id={puzzle_id} return_code={code} log_path={log_path}')
        return result
    if parsed is None:
        return {'puzzle_id': puzzle_id, 'solved': 0, 'seconds': elapsed, 'length': None, 'solution': '', 'return_code': code, 'log_path': str(log_path)}
    solved = int(parsed.group(1))
    return {
        'puzzle_id': int(parsed.group(2)),
        'solved': solved,
        'seconds': float(parsed.group(3)),
        'length': int(parsed.group(4)) if solved else None,
        'solution': parsed.group(5) if solved else '',
        'return_code': code,
        'log_path': str(log_path),
    }

results = []
for index, puzzle_id in enumerate(range(START_PUZZLE_ID, START_PUZZLE_ID + PUZZLE_COUNT), start=1):
    result = run_puzzle(puzzle_id, index)
    results.append(result)
    if PUZZLE_LOG_EVERY and (index % PUZZLE_LOG_EVERY == 0):
        print(f'puzzle_progress={index}/{PUZZLE_COUNT} puzzle_id={puzzle_id} solved={result["solved"]} seconds={result["seconds"]:.6f} length={result["length"]}')
    df = pd.DataFrame(results)
    df.to_csv(RESULTS_CSV, index=False)
    solved_df = df[df['solved'] == 1][['puzzle_id', 'solution']].rename(columns={'puzzle_id': 'initial_state_id', 'solution': 'path'})
    solved_df.to_csv(SUBMISSION_CSV, index=False)

pd.DataFrame(results)


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(results)
solved = df[df['solved'] == 1].copy()
lengths = [int(x) for x in solved['length'].dropna().tolist()]
hist_path = WORK_DIR / 'solution_length_histogram.png'
if lengths:
    min_len = min(lengths)
    max_len = max(lengths)
    avg_len = sum(lengths) / len(lengths)
    mode_len = Counter(lengths).most_common(1)[0][0]
    plt.figure(figsize=(10, 5))
    plt.hist(lengths, bins=range(min_len, max_len + 2), edgecolor='black')
    plt.xlabel('solution_length')
    plt.ylabel('solved_puzzle_count')
    plt.title('Solved puzzle solution lengths')
    plt.tight_layout()
    plt.savefig(hist_path, dpi=160)
    print(f'solved_count={len(lengths)} total_count={PUZZLE_COUNT}')
    print(f'min_solution_length={min_len}')
    print(f'max_solution_length={max_len}')
    print(f'avg_solution_length={avg_len:.6f}')
    print(f'mode_solution_length={mode_len}')
    print(f'histogram_png={hist_path}')
else:
    print(f'solved_count=0 total_count={PUZZLE_COUNT}')
    print(f'histogram_png_not_created={hist_path}')

solved
